DeepSeek Architecture

In [ ]:
# -*- coding: utf-8 -*-
"""DeepSeek-V3 Architecture with MLHA and MoE

Implements:
- Multi-Head Latent Attention (MLHA) with compression ratio
- Mixture of Experts (MoE) with loss-free load balancing
- Based on correct reference implementation
"""

import os
import math
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from typing import Optional, Tuple, Dict
from dataclasses import dataclass
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
work_dir = "/content/drive/MyDrive/ERAV4/Session_14"
os.chdir(work_dir)

# Verify files
print("Current directory:", os.getcwd())
print("Files in directory:", os.listdir())
print()

# Check if input.txt exists
if os.path.exists('input.txt'):
    print("✅ Found input.txt")
    with open('input.txt', 'r') as f:
        text = f.read()
    print(f"✅ Text file size: {len(text)} characters")
else:
    print("❌ input.txt not found!")
    print("Available files:", os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content/drive/MyDrive/ERAV4/Session_14
Files in directory: ['input.txt', 'checkpoints']

✅ Found input.txt
✅ Text file size: 1115394 characters


In [ ]:
# ============================================================================
# LAYER NORMALIZATION
# ============================================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""

    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x


In [ ]:

# ============================================================================
# ROTARY POSITIONAL EMBEDDING (RoPE)
# ============================================================================

class LlamaRotaryEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) - Note: Only half dimension for RoPE"""

    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        self.dim = dim  # This will be head_dim//2 for RoPE
        self.max_position_embeddings = max_position_embeddings
        self.base = base

        # Compute inverse frequencies
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Build position indices
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings,
            device=self.inv_freq.device,
            dtype=torch.get_default_dtype()
        )

    def _set_cos_sin_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype):
        self.max_seq_len_cached = seq_len
        t = torch.arange(self.max_seq_len_cached, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, seq_len: int, device: torch.device):
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=device, dtype=torch.get_default_dtype())

        return (
            self.cos_cached[:seq_len],
            self.sin_cached[:seq_len],
        )

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Apply rotary position embedding to query and key tensors."""
    # q, k: [batch_size, num_heads, seq_len, head_dim]
    # cos, sin: [seq_len, head_dim//2]
    cos = cos.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, head_dim//2]
    sin = sin.unsqueeze(0).unsqueeze(0)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
# ============================================================================
# MULTI-HEAD LATENT ATTENTION (MLHA)
# ============================================================================
class MultiHeadLatentAttention(nn.Module):
    """
    Multi-Head Latent Attention with low-rank factorization.
    Key difference: RoPE is applied to LATENT dimensions, not full dimensions.
    """

    def __init__(self, hidden_size: int, num_heads: int, compression_ratio: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.compression_ratio = compression_ratio
        self.latent_dim = hidden_size // compression_ratio

        # For proper reshaping, we need dimensions divisible by num_heads
        # Each head will get half the head_dim for content, half for position
        assert self.head_dim % 2 == 0, f"head_dim must be even, got {self.head_dim}"
        self.half_head_dim = self.head_dim // 2

        # Total dimensions for half projections
        self.kqv_half_dim = self.num_heads * self.half_head_dim

        # Compressed KV Projection (shared compression)
        self.kv_proj_d = nn.Linear(hidden_size, self.latent_dim, bias=False)

        # Compressed Q Projection
        self.q_proj_d = nn.Linear(hidden_size, self.latent_dim, bias=False)

        # Uncompress KQV Projections (output half dimensions)
        self.k_proj_u = nn.Linear(self.latent_dim, self.kqv_half_dim, bias=False)
        self.q_proj_u = nn.Linear(self.latent_dim, self.kqv_half_dim, bias=False)
        self.v_proj_u = nn.Linear(self.latent_dim, hidden_size, bias=False)

        # RoPE components (applied to LATENT dimensions)
        # K RoPE is built from X (uncompressed keys)
        self.rope_k = nn.Linear(hidden_size, self.kqv_half_dim, bias=False)
        # Q RoPE is built from q_proj_d (compressed queries)
        self.rope_q = nn.Linear(self.latent_dim, self.kqv_half_dim, bias=False)

        # Output projection
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        # RoPE Embeddings - Only half size for RoPE components
        self.rotary_emb = LlamaRotaryEmbedding(self.half_head_dim)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape

        # Compressed KV Projections
        kv_d = self.kv_proj_d(x)  # [bs, seq_len, latent_dim]

        # Compressed Q Projections
        q_d = self.q_proj_d(x)  # [bs, seq_len, latent_dim]

        # Uncompress KQV Projections
        k_proj_2 = self.k_proj_u(kv_d)  # [bs, seq_len, kqv_half_dim]
        q_proj_2 = self.q_proj_u(q_d)   # [bs, seq_len, kqv_half_dim]
        v = self.v_proj_u(kv_d)         # [bs, seq_len, hidden_size]

        # Generate RoPE Components
        # K is built from X, Q is built from q_d
        k_rope_2 = self.rope_k(x)       # [bs, seq_len, kqv_half_dim]
        q_rope_2 = self.rope_q(q_d)     # [bs, seq_len, kqv_half_dim]

        # Reshape components for heads before RoPE
        k_proj_2 = k_proj_2.view(batch_size, seq_len, self.num_heads, self.half_head_dim)
        k_rope_2 = k_rope_2.view(batch_size, seq_len, self.num_heads, self.half_head_dim)
        q_proj_2 = q_proj_2.view(batch_size, seq_len, self.num_heads, self.half_head_dim)
        q_rope_2 = q_rope_2.view(batch_size, seq_len, self.num_heads, self.half_head_dim)

        # Transpose for RoPE application: [bs, num_heads, seq_len, half_head_dim]
        k_proj_2 = k_proj_2.transpose(1, 2)
        k_rope_2 = k_rope_2.transpose(1, 2)
        q_proj_2 = q_proj_2.transpose(1, 2)
        q_rope_2 = q_rope_2.transpose(1, 2)

        # Apply RoPE to positional aware components
        cos, sin = self.rotary_emb(seq_len, x.device)
        k_rope_2, _ = apply_rotary_emb(k_rope_2, k_rope_2, cos, sin)
        q_rope_2, _ = apply_rotary_emb(q_rope_2, q_rope_2, cos, sin)

        # Concatenate content and position: [bs, num_heads, seq_len, head_dim]
        k = torch.cat([k_proj_2, k_rope_2], dim=-1)
        q = torch.cat([q_proj_2, q_rope_2], dim=-1)

        # Reshape V: [bs, num_heads, seq_len, head_dim]
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_output = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=True
        )

        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.hidden_size)

        return self.o_proj(attn_output)

In [ ]:
# ============================================================================
# MIXTURE OF EXPERTS (MoE)
# ============================================================================

class SiLUActivation(nn.Module):
    """SiLU (Swish) activation function"""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.silu(x)

class DeepSeekExpertLayer(nn.Module):
    """Single Expert in MoE - Standard FFN"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = SiLUActivation()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: down(act(gate(x)) * up(x))
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

class DeepSeekMoE(nn.Module):
    """
    Mixture of Experts with Loss-Free Load Balancing.
    Following the exact reference implementation structure.
    """

    def __init__(self, hidden_size: int, intermediate_size: int,
                 num_experts: int = 8, num_shared_experts: int = 1, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.num_shared_experts = num_shared_experts
        self.num_routed_experts = num_experts - num_shared_experts
        self.top_k = top_k
        self.hidden_size = hidden_size

        # Shared experts
        self.shared_experts = nn.ModuleList([
            DeepSeekExpertLayer(hidden_size, intermediate_size)
            for _ in range(self.num_shared_experts)
        ])

        # Routed experts
        self.routed_experts = nn.ModuleList([
            DeepSeekExpertLayer(hidden_size, intermediate_size)
            for _ in range(self.num_routed_experts)
        ])

        # Router components
        self.router = nn.Linear(hidden_size, self.num_routed_experts, bias=False)
        self.routing_bias = nn.Parameter(torch.zeros(self.num_routed_experts))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, hidden_size = x.shape

        # Process through shared experts
        shared_output = sum(expert(x) for expert in self.shared_experts)
        if self.num_shared_experts > 1:
            shared_output = shared_output / self.num_shared_experts

        # Calculate routing scores
        routing_logits = self.router(x) + self.routing_bias

        # Get top-k experts per token (using sigmoid before topk as in reference)
        routing_probs = torch.sigmoid(routing_logits)
        scores, indices = torch.topk(routing_probs, self.top_k, dim=-1)

        # Normalize the top-k scores
        scores = scores / scores.sum(dim=-1, keepdim=True)

        # Process through selected experts
        combined_output = torch.zeros_like(x)
        for k in range(self.top_k):
            expert_indices = indices[..., k]
            expert_scores = scores[..., k:k+1]

            # Process each expert
            for i in range(self.num_routed_experts):
                mask = (expert_indices == i)
                if mask.any():
                    expert_input = x[mask]
                    expert_output = self.routed_experts[i](expert_input)
                    combined_output[mask] += expert_output * expert_scores[mask]

        # Combine shared and routed outputs
        final_output = shared_output + combined_output

        return final_output

    def update_bias_terms(self, expert_load: torch.Tensor):
        """
        Update routing bias based on expert load.
        expert_load: tensor of shape [num_routed_experts] with load per expert
        """
        # Target load is uniform distribution
        target_load = 1.0 / self.num_routed_experts
        load_diff = expert_load - target_load

        # Dynamic update rate based on the magnitude of load imbalance
        update_rate = 0.1 * torch.abs(load_diff)

        # Update the routing bias using the dynamic update rate
        with torch.no_grad():
            self.routing_bias.data -= update_rate * load_diff


In [ ]:
# ============================================================================
# DECODER LAYER
# ============================================================================

class LlamaDecoderLayer(nn.Module):
    """Transformer decoder layer with MLHA and MoE"""

    def __init__(self, hidden_size: int, num_heads: int, intermediate_size: int,
                 compression_ratio: int, num_experts: int, num_shared_experts: int, top_k: int):
        super().__init__()
        self.hidden_size = hidden_size

        # MLHA instead of standard attention
        self.self_attn = MultiHeadLatentAttention(hidden_size, num_heads, compression_ratio)

        # Layer norms
        self.input_layernorm = RMSNorm(hidden_size)
        self.post_attention_layernorm = RMSNorm(hidden_size)

        # MoE instead of standard MLP
        self.mlp = DeepSeekMoE(hidden_size, intermediate_size,
                               num_experts, num_shared_experts, top_k)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Self-attention
        residual = x
        x = self.self_attn(self.input_layernorm(x), attention_mask)
        x = x + residual

        # Feedforward
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = x + residual

        return x

In [ ]:
# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

class DeepSeekConfig:
    """Configuration class for DeepSeek-V3"""

    def __init__(
        self,
        vocab_size: int = 49152,
        hidden_size: int = 576,
        intermediate_size: int = 1536,
        num_hidden_layers: int = 30,
        num_attention_heads: int = 9,
        max_position_embeddings: int = 2048,
        rms_norm_eps: float = 1e-5,
        rope_theta: float = 10000.0,
        compression_ratio: int = 8,
        num_experts: int = 8,
        num_shared_experts: int = 1,
        top_k_experts: int = 2,
        pad_token_id: int = 0,
        bos_token_id: int = 1,
        eos_token_id: int = 2,
        tie_word_embeddings: bool = True,
        **kwargs
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.compression_ratio = compression_ratio
        self.num_experts = num_experts
        self.num_shared_experts = num_shared_experts
        self.top_k_experts = top_k_experts
        self.pad_token_id = pad_token_id
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.tie_word_embeddings = tie_word_embeddings

        # Validate dimensions for MLHA
        head_dim = hidden_size // num_attention_heads
        assert head_dim % 2 == 0, (
            f"head_dim must be even for MLHA. Got hidden_size={hidden_size}, "
            f"num_heads={num_attention_heads}, head_dim={head_dim}. "
            f"Try hidden_size=768, num_heads=12 (head_dim=64) or "
            f"hidden_size=768, num_heads=8 (head_dim=96)"
        )

In [ ]:
# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class DeepSeekModel(nn.Module):
    """DeepSeek-V3 Base Model"""

    def __init__(self, config: DeepSeekConfig):
        super().__init__()
        self.config = config
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(
                config.hidden_size,
                config.num_attention_heads,
                config.intermediate_size,
                config.compression_ratio,
                config.num_experts,
                config.num_shared_experts,
                config.top_k_experts
            ) for _ in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Embed tokens
        hidden_states = self.embed_tokens(input_ids)

        # Apply transformer layers
        for decoder_layer in self.layers:
            hidden_states = decoder_layer(hidden_states, attention_mask)

        # Final layer norm
        hidden_states = self.norm(hidden_states)

        return hidden_states

class DeepSeekForCausalLM(nn.Module):
    """DeepSeek-V3 model with language modeling head"""

    def __init__(self, config: DeepSeekConfig):
        super().__init__()
        self.config = config
        self.model = DeepSeekModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Tie weights between embedding and lm_head if specified
        if config.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.LongTensor] = None,
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
        # Forward pass through model
        hidden_states = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Compute logits
        logits = self.lm_head(hidden_states)

        # Compute loss if labels provided
        loss = None
        if labels is not None:
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # Flatten the tokens
            loss_fct = nn.CrossEntropyLoss()
            shift_logits = shift_logits.view(-1, self.config.vocab_size)
            shift_labels = shift_labels.view(-1)

            # Enable model parallelism
            shift_labels = shift_labels.to(shift_logits.device)
            loss = loss_fct(shift_logits, shift_labels)

        return loss, logits

    def generate(
        self,
        input_ids: torch.LongTensor,
        max_new_tokens: int = 50,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        top_p: Optional[float] = None,
    ) -> torch.LongTensor:
        """Simple generation with sampling"""
        self.eval()

        for _ in range(max_new_tokens):
            # Forward pass
            with torch.no_grad():
                _, logits = self.forward(input_ids)

            # Get logits for last token
            next_token_logits = logits[:, -1, :] / temperature

            # Apply top-k filtering
            if top_k is not None:
                indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                next_token_logits[indices_to_remove] = float('-inf')

            # Apply top-p (nucleus) filtering
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                next_token_logits[indices_to_remove] = float('-inf')

            # Sample next token
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Append to sequence
            input_ids = torch.cat([input_ids, next_token], dim=-1)

            # Stop if EOS token generated
            if next_token.item() == self.config.eos_token_id:
                break

        return input_ids

    def calculate_expert_load(self, input_ids: torch.Tensor) -> Dict[int, torch.Tensor]:
        """Calculate expert load for each layer for load balancing"""
        expert_loads = {}

        # Track routing decisions for each layer
        for layer_idx, layer in enumerate(self.model.layers):
            if hasattr(layer.mlp, 'routed_experts'):
                # Calculate expert load
                expert_load = torch.zeros(layer.mlp.num_routed_experts, device=input_ids.device)

                # Get routing logits
                hidden_states = self.model.embed_tokens(input_ids)
                for i in range(layer_idx + 1):
                    hidden_states = self.model.layers[i](hidden_states)

                routing_logits = layer.mlp.router(hidden_states) + layer.mlp.routing_bias
                routing_probs = torch.sigmoid(routing_logits)
                _, indices = torch.topk(routing_probs, layer.mlp.top_k, dim=-1)

                # Count expert usage
                for i in range(layer.mlp.num_routed_experts):
                    expert_load[i] = (indices == i).sum().float()

                # Normalize by total tokens
                expert_load = expert_load / (input_ids.size(0) * input_ids.size(1) * layer.mlp.top_k)
                expert_loads[layer_idx] = expert_load

        return expert_loads

    def update_expert_load_balancing(self, input_ids: torch.Tensor):
        """Update routing bias for all MoE layers based on current batch"""
        expert_loads = self.calculate_expert_load(input_ids)

        for layer_idx, expert_load in expert_loads.items():
            self.model.layers[layer_idx].mlp.update_bias_terms(expert_load)

def count_parameters(model):
    """Count total and trainable parameters"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

In [ ]:

# ============================================================================
# TEST MODEL
# ============================================================================

if __name__ == "__main__":
    # Test model initialization
    config = DeepSeekConfig(
    vocab_size=49152,
    hidden_size=576,
    num_hidden_layers=30,
    num_attention_heads=9,
    intermediate_size=1536,      # SAME as SmolLM

    # DeepSeek MoE but SMALL
    num_experts=2,
    num_shared_experts=1,
    top_k_experts=1,

    # Apply smaller FFN only inside MoE
    moe_intermediate_size=1024,  # NEW HYPERPARAM
    compression_ratio=8,         # MLHA stays

    max_position_embeddings=2048,
    rope_theta=10000.0,
    tie_word_embeddings=True,
)


    model = DeepSeekForCausalLM(config)

    total_params, trainable_params = count_parameters(model)
    print(f"DeepSeek-V3 Model initialized")
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
    print(f"Trainable parameters: {trainable_params:,}")

    # Test forward pass
    batch_size = 2
    seq_length = 32
    input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_length))

    print(f"\nTesting forward pass with input shape: {input_ids.shape}")
    loss, logits = model(input_ids, labels=input_ids)
    print(f"Output logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")
    print("\nDeepSeek-V3 architecture verified successfully!")

DeepSeek-V3 Model initialized
Total parameters: 208,145,118 (208.1M)
Trainable parameters: 208,145,118

Testing forward pass with input shape: torch.Size([2, 32])
Output logits shape: torch.Size([2, 32, 49152])
Loss: 10.8110

DeepSeek-V3 architecture verified successfully!


In [ ]:
# -*- coding: utf-8 -*-
"""DeepSeek-V3 Training Script

Training script with:
- MLHA (compression ratio 8)
- MoE with 8 experts, 1 shared, top-k=2
- Loss-free load balancing
- All speed-ups enabled
"""

import os
import math
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import AutoTokenizer

In [ ]:
# Device configuration
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(f"Using device: {device}")

# Set seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

Using device: cuda


In [ ]:

class DataLoaderLite:
    """Data loader that loads tokens from text file"""

    def __init__(self, B, T, data_path='input.txt'):
        self.B = B
        self.T = T

        # Load tokens from disk
        with open(data_path, 'r') as f:
            text = f.read()

        # Use SmolLM2 tokenizer (compatible vocab size)
        enc = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
        tokens = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f'Loaded {len(self.tokens)} tokens')
        print(f'Batch size = {B * T} tokens')
        print(f'1 Step = {len(self.tokens) // (B * T)} batches\n')

        # State
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position: self.current_position + B * T + 1]
        x = (buf[:-1]).view(B, T)  # inputs
        y = (buf[1:]).view(B, T)   # targets

        # Advance position
        self.current_position += B * T

        # Reset if out of bounds
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0

        return x, y

In [ ]:
# ============================================================================
# TRAINING SETUP
# ============================================================================

# Model configuration (as per assignment)

# config = DeepSeekConfig(
#     vocab_size=49152,
#     hidden_size=576,
#     intermediate_size=1536,
#     num_hidden_layers=30,
#     num_attention_heads=9,
#     max_position_embeddings=2048,
#     compression_ratio=8,
#     num_experts=8,
#     num_shared_experts=1,
#     top_k_experts=2,
#     rms_norm_eps=1e-5,
#     rope_theta=10000.0,
#     tie_word_embeddings=True,
# )

import gc
gc.collect()
torch.cuda.empty_cache()

torch.set_default_dtype(torch.bfloat16)  # all new tensors use bf16
torch.set_float32_matmul_precision('high')

config = DeepSeekConfig(
    vocab_size=49152,
    hidden_size=576,
    num_hidden_layers=30,
    num_attention_heads=9,
    intermediate_size=1536,      # SAME as SmolLM

    # DeepSeek MoE but SMALL
    num_experts=2,
    num_shared_experts=1,
    top_k_experts=1,

    # Apply smaller FFN only inside MoE
    moe_intermediate_size=1024,  # NEW HYPERPARAM
    compression_ratio=8,         # MLHA stays

    max_position_embeddings=2048,
    rope_theta=10000.0,
    tie_word_embeddings=True,
)


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
# Initialize model
print("Initializing DeepSeek-V3 model...")
torch.set_float32_matmul_precision('high')  # Speedup 1
# model = DeepSeekForCausalLM(config)
# model.to(device)
# model = torch.compile(model)  # Speedup 3

model = DeepSeekForCausalLM(config)
model = model.to(device, dtype=torch.bfloat16)  # move in bf16
model = torch.compile(model)  # compile AFTER bf16, AFTER to(device)


# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")

# Data loader
train_loader = DataLoaderLite(B=2, T=512)

# Optimizer configuration
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1
)

# Training configuration
total_steps = 10000
warmup_steps = 1000  # 10% warmup
save_interval = 500
balance_interval = 200  # Update expert balancing every 200 steps

# Learning rate scheduler
max_steps = total_steps
min_lr = 5e-5

Initializing DeepSeek-V3 model...
Total parameters: 208,145,118 (208.1M)


Token indices sequence length is longer than the specified maximum sequence length for this model (341094 > 8192). Running this sequence through the model will result in indexing errors


Loaded 341094 tokens
Batch size = 1024 tokens
1 Step = 333 batches



In [ ]:
def get_lr(step):
    """Cosine annealing with warmup"""
    # Warmup phase
    if step < warmup_steps:
        return 5e-4 * step / warmup_steps

    # Cosine annealing phase
    if step > max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (5e-4 - min_lr)

# Tokenizer for generation
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

In [ ]:
# ============================================================================
# TRAINING LOOP
# ============================================================================

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80 + "\n")

# Create checkpoint directory
os.makedirs('checkpoints', exist_ok=True)

# Training loop
for step in range(1, total_steps + 1):
    t0 = time.time()

    # Get batch
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)

    # Forward pass with mixed precision
    optimizer.zero_grad()
    with torch.autocast(device_type=device, dtype=torch.bfloat16):  # Speedup 2
        loss, logits = model(x, labels=y)

    # Backward pass
    loss.backward()

    # Update learning rate
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # Optimizer step
    optimizer.step()

    # Update expert load balancing (loss-free)
    if step % balance_interval == 0:
        model.update_expert_load_balancing(x)

    # Synchronize for accurate timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)

    # Print progress
    print(f'[TRAIN] Step {step:5d}/{total_steps} | Loss: {loss.item():.4f} | LR: {lr:.6f} | dt: {dt:6.2f}ms | tok/sec: {tokens_per_sec:7.2f}')

    # Save checkpoint periodically
    if step % save_interval == 0:
        checkpoint_path = f'checkpoints/checkpoint_step_{step}.pth'
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'config': config.__dict__,
        }, checkpoint_path)
        print(f'[SAVE] Checkpoint saved at step {step}')

        # Generate sample output
        prompt = "To be or not to be"
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        model.eval()
        with torch.no_grad():
            output = model.generate(input_ids, max_new_tokens=30, temperature=0.8)
        model.train()

        generated_text = tokenizer.decode(output[0])
        print(f'[SAMPLE] "{generated_text}"')
        print()

print("\n" + "="*80)
print("TRAINING COMPLETED!")
print("="*80)


STARTING TRAINING



W1123 16:30:55.203000 8588 torch/_inductor/utils.py:1558] [2/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dis

Streaming output truncated to the last 5000 lines.
[TRAIN] Step  5070/10000 | Loss: 2.9292 | LR: 0.000309 | dt: 716.14ms | tok/sec: 1429.89
[TRAIN] Step  5071/10000 | Loss: 2.6156 | LR: 0.000309 | dt: 710.54ms | tok/sec: 1441.16
[TRAIN] Step  5072/10000 | Loss: 3.1974 | LR: 0.000308 | dt: 711.97ms | tok/sec: 1438.27
[TRAIN] Step  5073/10000 | Loss: 2.4753 | LR: 0.000308 | dt: 756.45ms | tok/sec: 1353.70
[TRAIN] Step  5074/10000 | Loss: 2.8810 | LR: 0.000308 | dt: 752.53ms | tok/sec: 1360.74
[TRAIN] Step  5075/10000 | Loss: 3.1552 | LR: 0.000308 | dt: 737.90ms | tok/sec: 1387.73
[TRAIN] Step  5076/10000 | Loss: 2.7238 | LR: 0.000308 | dt: 754.34ms | tok/sec: 1357.48
[TRAIN] Step  5077/10000 | Loss: 3.5482 | LR: 0.000308 | dt: 734.78ms | tok/sec: 1393.62
[TRAIN] Step  5078/10000 | Loss: 3.5967 | LR: 0.000308 | dt: 709.82ms | tok/sec: 1442.61
[TRAIN] Step  5079/10000 | Loss: 2.6831 | LR: 0.000308 | dt: 706.78ms | tok/sec: 1448.82
[TRAIN] Step  5080/10000 | Loss: 2.6565 | LR: 0.000308 | dt

In [ ]:
# ============================================================================
# FINAL CHECKPOINT
# ============================================================================

# Save final model
final_checkpoint_path = 'checkpoints/final_model.pth'
torch.save({
    'step': total_steps,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
    'config': config.__dict__,
}, final_checkpoint_path)
print(f'\n[SAVE] Final model saved to {final_checkpoint_path}')
print(f'[INFO] Final loss after {total_steps} steps: {loss.item():.4f}')



[SAVE] Final model saved to checkpoints/final_model.pth
[INFO] Final loss after 10000 steps: 1.7221


In [ ]:
# ============================================================================
# GENERATE 5 BEST OUTPUTS
# ============================================================================

print("\n" + "="*80)
print("GENERATING 5 SAMPLE OUTPUTS")
print("="*80 + "\n")

model.eval()

prompts = [
    "Once upon a time",
    "The meaning of life is",
    "In a world where",
    "To be or not to be",
    "The future of AI"
]

outputs = []

for i, prompt in enumerate(prompts, 1):
    print(f"\nPrompt {i}: {prompt}")
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=50,
            temperature=0.8,
            top_k=50,
            top_p=0.9
        )

    generated_text = tokenizer.decode(output[0])
    outputs.append(generated_text)
    print(f"Output: {generated_text}\n")
    print("-" * 80)

# Save outputs to file
with open('outputs.txt', 'w') as f:
    f.write("DeepSeek-V3 Generated Outputs\n")
    f.write("=" * 80 + "\n\n")
    for i, (prompt, output) in enumerate(zip(prompts, outputs), 1):
        f.write(f"Prompt {i}: {prompt}\n")
        f.write(f"Output: {output}\n")
        f.write("-" * 80 + "\n\n")

print("\n[SAVE] Outputs saved to outputs.txt")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)


GENERATING 5 SAMPLE OUTPUTS


Prompt 1: Once upon a time
Output: Once upon a time
 best and fair and thing by wion
 than want thousand one;
, the of that have of
 shall and him the, less but
's as were
 our: we your,; that buttis
's, I beul

--------------------------------------------------------------------------------

Prompt 2: The meaning of life is
Output: The meaning of life is the of.
L I, night. is for rest here the.
Second of noble is and for fl
OL of noble but that didly: his
orthy the:' I him,, l, all all world
an;

--------------------------------------------------------------------------------

Prompt 3: In a world where
Output: In a world where should call their with;And
 his queen the fl, oer oer'd,ed; way be'd
 many proud he.
MAN
ere you;, all once'd it yours I.
VGle off What the

--------------------------------------------------------------------------------

Prompt 4: To be or not to be
Output: To be or not to be'
 much.
MR very on head go
 queen what pleased what 